# Clipping WCS Annual Burn Probability to San Diego County

This notebook demonstrates how to:
1. Fetch San Diego County boundary from a WFS service
2. Download annual burn probability data from a WCS service
3. Handle coordinate reference system transformations (EPSG:4269 to EPSG:3310)
4. Clip the raster data to the county boundary
5. Save the result as a GeoTIFF file
6. Visualize the clipped result

## Install Required Dependencies

In [ ]:
! pip install rasterio geopandas shapely pyproj pillow requests matplotlib -q

## Import Libraries

In [ ]:
import requests
import rasterio
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.transform import from_bounds
import geopandas as gpd
from shapely.geometry import shape, box
import numpy as np
from pyproj import CRS, Transformer
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

## Configuration

In [ ]:
# Service endpoints
WFS_URL = "https://sparcal.sdsc.edu/geoserver/boundary/wfs"
WCS_URL = "https://sparcal.sdsc.edu/geoserver/rrk/wcs"

# Layer names
WFS_LAYER = "boundary:ca_counties"
WCS_LAYER = "rrk__annualburnprobability_202212_202406_t1_v5"

# Coordinate reference systems
CRS_WFS = "EPSG:4269"  # NAD83
CRS_WCS = "EPSG:3310"  # California Albers

# Output file
OUTPUT_FILE = "san_diego_burn_probability.tif"

print("Configuration loaded successfully")

## Step 1: Fetch San Diego County Boundary from WFS

In [ ]:
from shapely.geometry import shape
def get_san_diego_boundary():
    """Fetch San Diego County boundary from WFS service."""
    print("Fetching San Diego County boundary from WFS...")
    
    # WFS GetFeature request for San Diego County
    params = {
        'service': 'WFS',
        'version': '2.0.0',
        'request': 'GetFeature',
        'typeNames': WFS_LAYER,
        'outputFormat': 'application/json',
        'CQL_FILTER': "name='San Diego'"
    }
    
    response = requests.get(WFS_URL, params=params)
    response.raise_for_status()
    
    data = response.json()
    
    if len(data['features']) == 0:
        raise ValueError("No features found for San Diego County")
    
    # Get the first feature (should be San Diego County)
    feature = data['features'][0]
    
    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(
        {'geometry': [shape(feature['geometry'])]},
        crs=CRS_WFS
    )
    
    print(f"  Fetched San Diego County boundary")
    print(f"  Geometry type: {gdf.geometry.iloc[0].geom_type}")
    print(f"  Bounds (EPSG:4269): {gdf.total_bounds}")
    
    return gdf

boundary_4269 = get_san_diego_boundary()

## Step 2: Transform Bounding Box to WCS CRS

In [ ]:
def transform_bbox(bbox, from_crs, to_crs):
    """Transform bounding box from one CRS to another."""
    minx, miny, maxx, maxy = bbox
    
    transformer = Transformer.from_crs(from_crs, to_crs, always_xy=True)
    
    # Transform all four corners
    corners = [
        transformer.transform(minx, miny),
        transformer.transform(maxx, miny),
        transformer.transform(minx, maxy),
        transformer.transform(maxx, maxy)
    ]
    
    # Get min/max of transformed coordinates
    x_coords = [c[0] for c in corners]
    y_coords = [c[1] for c in corners]
    
    return (min(x_coords), min(y_coords), max(x_coords), max(y_coords))

# Get bounding box in WFS CRS
bbox_4269 = boundary_4269.total_bounds

# Transform to WCS CRS
bbox_3310 = transform_bbox(bbox_4269, CRS_WFS, CRS_WCS)

print(f"\nTransformed bounding box to EPSG:3310: {bbox_3310}")

## Step 3: Fetch WCS Data for Bounding Box

In [ ]:
def get_wcs_data(bbox, width=None, height=None):
    """Fetch WCS data for a given bounding box."""
    print(f"\nFetching WCS data for bounding box (EPSG:3310): {bbox}")
    
    # Note: WCS 2.0.1 uses 'subset' parameter with uppercase axis labels
    # We need to pass multiple subset parameters
    # Try without scaleSize first to avoid server issues
    params = [
        ('service', 'WCS'),
        ('version', '2.0.1'),
        ('request', 'GetCoverage'),
        ('coverageId', WCS_LAYER),
        ('format', 'image/tiff'),
        ('subset', f'X({bbox[0]},{bbox[2]})'),
        ('subset', f'Y({bbox[1]},{bbox[3]})'),
        ('crs', 'http://www.opengis.net/def/crs/EPSG/0/3310')
    ]
    
    response = requests.get(WCS_URL, params=params)
    
    if response.status_code != 200:
        print(f"  Status code: {response.status_code}")
        print(f"  Response text: {response.text[:500]}")
        raise ValueError(f"WCS request failed with status {response.status_code}")
    
    print(f"  Successfully fetched WCS data ({len(response.content)} bytes)")
    
    # Read the GeoTIFF from memory
    import io
    with rasterio.open(io.BytesIO(response.content)) as src:
        data = src.read(1)
        transform = src.transform
        crs = src.crs
        nodata = src.nodata
        
        print(f"  Data shape: {data.shape}")
        print(f"  Data type: {data.dtype}")
        print(f"  CRS: {crs}")
        print(f"  Transform: {transform}")
        print(f"  NoData value: {nodata}")
        
        # Get data range (excluding nodata)
        valid_data = data[data != nodata]
        if len(valid_data) > 0:
            print(f"  Data range: {valid_data.min():.4f} to {valid_data.max():.4f}")
    
    return data, transform, crs, nodata

# Fetch WCS data for San Diego County bounding box
data, transform, crs, nodata = get_wcs_data(bbox_3310)

## Step 4: Clip Data to San Diego County Boundary

In [ ]:
def clip_to_boundary(data, transform, crs, boundary_gdf, nodata):
    """Clip raster data to boundary geometry."""
    print("\nClipping data to San Diego County boundary...")
    
    # Transform boundary to raster CRS
    boundary_raster_crs = boundary_gdf.to_crs(crs)
    
    print(f"  Boundary bounds in raster CRS: {boundary_raster_crs.total_bounds}")
    
    # Create a temporary in-memory raster
    from rasterio.io import MemoryFile
    
    with MemoryFile() as memfile:
        with memfile.open(
            driver='GTiff',
            height=data.shape[0],
            width=data.shape[1],
            count=1,
            dtype=data.dtype,
            crs=crs,
            transform=transform,
            nodata=nodata
        ) as dataset:
            dataset.write(data, 1)
        
        # Reopen for reading
        with memfile.open() as src:
            # Clip using the boundary geometry
            out_image, out_transform = mask(
                src,
                boundary_raster_crs.geometry,
                crop=True,
                nodata=nodata
            )
            out_image = out_image[0]  # Get first band
    
    # Calculate percentage of pixels within boundary
    total_pixels = np.prod(data.shape)
    valid_pixels = np.sum(out_image != nodata)
    percentage = (valid_pixels / total_pixels) * 100
    
    print(f"  Pixels within boundary: {valid_pixels} / {total_pixels} ({percentage:.1f}%)")
    
    return out_image, out_transform

# Clip the data
clipped_data, clipped_transform = clip_to_boundary(data, transform, crs, boundary_4269, nodata)

## Step 5: Save Clipped Data as GeoTIFF

In [ ]:
def save_geotiff(data, transform, crs, nodata, filename):
    """Save raster data as GeoTIFF file."""
    print(f"\nSaving clipped data to {filename}...")
    
    with rasterio.open(
        filename,
        'w',
        driver='GTiff',
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype=data.dtype,
        crs=crs,
        transform=transform,
        nodata=nodata,
        compress='lzw'
    ) as dst:
        dst.write(data, 1)
    
    print(f"  Successfully saved GeoTIFF")

# Save the clipped data
save_geotiff(clipped_data, clipped_transform, crs, nodata, OUTPUT_FILE)

## Step 6: Visualize the Clipped Result

In [ ]:
def visualize_clipped_result(clipped_data, clipped_transform, crs, boundary_gdf, nodata):
    """Visualize the clipped burn probability data."""
    print("\nCreating visualizations...")
    
    # Create figure with two subplots
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    # Create a custom colormap for burn probability (yellow to red)
    colors = ['#ffffcc', '#ffeda0', '#fed976', '#feb24c', '#fd8d3c', '#fc4e2a', '#e31a1c', '#bd0026', '#800026']
    cmap = LinearSegmentedColormap.from_list('burn_prob', colors, N=256)
    
    # Mask nodata values for visualization
    display_data = np.ma.masked_where(clipped_data == nodata, clipped_data)
    
    # Get extent for plotting
    height, width = clipped_data.shape
    extent = [
        clipped_transform[2],
        clipped_transform[2] + width * clipped_transform[0],
        clipped_transform[5] + height * clipped_transform[4],
        clipped_transform[5]
    ]
    
    # Plot 1: In original WCS CRS (EPSG:3310)
    ax1 = axes[0]
    im1 = ax1.imshow(display_data, extent=extent, cmap=cmap, origin='upper')
    
    # Add boundary in WCS CRS
    boundary_wcs = boundary_gdf.to_crs(crs)
    boundary_wcs.boundary.plot(ax=ax1, edgecolor='black', linewidth=2, facecolor='none')
    
    ax1.set_title('Annual Burn Probability - San Diego County\n(EPSG:3310 - California Albers)', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Easting (meters)', fontsize=12)
    ax1.set_ylabel('Northing (meters)', fontsize=12)
    cbar1 = plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
    cbar1.set_label('Burn Probability', fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Reprojected to EPSG:4269 (NAD83)
    ax2 = axes[1]
    
    # Reproject the clipped data to EPSG:4269
    from rasterio.warp import reproject, Resampling
    
    # Calculate destination transform and dimensions
    dst_crs = CRS.from_epsg(4269)
    dst_transform, dst_width, dst_height = calculate_default_transform(
        crs, dst_crs, width, height, left=extent[0], bottom=extent[2], right=extent[1], top=extent[3]
    )
    
    # Create destination array
    dst_data = np.zeros((dst_height, dst_width), dtype=clipped_data.dtype)
    
    # Reproject
    reproject(
        source=clipped_data,
        destination=dst_data,
        src_transform=clipped_transform,
        src_crs=crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.bilinear,
        src_nodata=nodata,
        dst_nodata=nodata
    )
    
    # Mask nodata values for visualization
    display_data_4269 = np.ma.masked_where(dst_data == nodata, dst_data)
    
    # Get extent for reprojected data
    extent_4269 = [
        dst_transform[2],
        dst_transform[2] + dst_width * dst_transform[0],
        dst_transform[5] + dst_height * dst_transform[4],
        dst_transform[5]
    ]
    
    im2 = ax2.imshow(display_data_4269, extent=extent_4269, cmap=cmap, origin='upper')
    
    # Add boundary in EPSG:4269
    boundary_gdf.boundary.plot(ax=ax2, edgecolor='black', linewidth=2, facecolor='none')
    
    ax2.set_title('Annual Burn Probability - San Diego County\n(EPSG:4269 - NAD83)', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Longitude', fontsize=12)
    ax2.set_ylabel('Latitude', fontsize=12)
    cbar2 = plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
    cbar2.set_label('Burn Probability', fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('san_diego_burn_probability_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("  Visualization saved as 'san_diego_burn_probability_visualization.png'")
    
    # Print statistics
    valid_data = clipped_data[clipped_data != nodata]
    if len(valid_data) > 0:
        print(f"\n  Statistics for clipped data:")
        print(f"    Min: {valid_data.min():.6f}")
        print(f"    Max: {valid_data.max():.6f}")
        print(f"    Mean: {valid_data.mean():.6f}")
        print(f"    Std: {valid_data.std():.6f}")
        print(f"    Valid pixels: {len(valid_data)}")

# Create visualizations
visualize_clipped_result(clipped_data, clipped_transform, crs, boundary_4269, nodata)

## Summary

In [ ]:
print("\n" + "="*70)
print("SUCCESS! Script completed successfully.")
print(f"Output file: {OUTPUT_FILE}")
print("="*70)

print("\nWhat was accomplished:")
print("1. ✓ Fetched San Diego County boundary from WFS (EPSG:4269)")
print("2. ✓ Transformed bounding box to WCS CRS (EPSG:3310)")
print("3. ✓ Downloaded annual burn probability data from WCS")
print("4. ✓ Clipped raster data to county boundary")
print("5. ✓ Saved result as GeoTIFF file")
print("6. ✓ Created visualizations in both CRS systems")